# EDA: Sales Order Analysis

**Purpose:** Exploratory data analysis on the Silver sales orders table.  
**Audience:** Data analysts and business stakeholders.  
**Last Updated:** 2024-01-20  

This notebook provides an overview of:
- Revenue trends over time
- Top customers and products
- Regional performance
- Order status distribution


In [ ]:
from pyspark.sql import functions as F
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

SILVER_TABLE = 'contoso_analytics_lakehouse.silver.sales_orders_cleansed'

In [ ]:
# Load current Silver data
df = (
    spark.table(SILVER_TABLE)
    .filter("_is_current = true AND status = 'Completed'")
    .select(
        'order_id', 'order_date', 'order_year', 'order_month',
        'customer_id', 'customer_name', 'product_id', 'product_name',
        'category', 'region', 'sales_rep_id',
        'quantity', 'unit_price_usd', 'total_amount_usd'
    )
    .toPandas()
)

df['order_date'] = pd.to_datetime(df['order_date'])
df['year_month'] = df['order_date'].dt.to_period('M')
print(f'Loaded {len(df):,} completed orders')

In [ ]:
# ── Monthly revenue trend ──────────────────────────────────────────────────────
monthly_rev = (
    df.groupby('year_month')['total_amount_usd']
    .sum()
    .reset_index()
    .sort_values('year_month')
)
monthly_rev['year_month_str'] = monthly_rev['year_month'].astype(str)

fig, ax = plt.subplots()
ax.bar(monthly_rev['year_month_str'], monthly_rev['total_amount_usd'], color='steelblue')
ax.set_title('Monthly Revenue (Completed Orders, USD)', fontsize=14)
ax.set_xlabel('Month')
ax.set_ylabel('Revenue (USD)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# ── Top 10 customers by revenue ────────────────────────────────────────────────
top_customers = (
    df.groupby('customer_name')['total_amount_usd']
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

fig, ax = plt.subplots()
top_customers.plot(kind='barh', ax=ax, color='teal')
ax.set_title('Top 10 Customers by Revenue (USD)', fontsize=14)
ax.set_xlabel('Total Revenue (USD)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# ── Revenue by region and category ────────────────────────────────────────────
region_cat = (
    df.groupby(['region', 'category'])['total_amount_usd']
    .sum()
    .unstack(fill_value=0)
)

region_cat.plot(kind='bar', stacked=True, colormap='Set2')
plt.title('Revenue by Region and Product Category', fontsize=14)
plt.xlabel('Region')
plt.ylabel('Revenue (USD)')
plt.xticks(rotation=0)
plt.gca().yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.legend(title='Category', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# ── Summary statistics ─────────────────────────────────────────────────────────
print('=== Summary Statistics ===')
print(f"Total completed revenue (USD): ${df['total_amount_usd'].sum():>15,.2f}")
print(f"Number of orders             : {len(df):>15,}")
print(f"Unique customers             : {df['customer_id'].nunique():>15,}")
print(f"Unique products              : {df['product_id'].nunique():>15,}")
print(f"Average order value (USD)    : ${df['total_amount_usd'].mean():>15,.2f}")
print(f"Date range                   : {df['order_date'].min().date()} → {df['order_date'].max().date()}")